# 批量问答：原始 PDF → Markdown → 建库 → 回答问题集 → 整理保存

运行本 Notebook（从上到下依次执行所有 cell）即可完成：

1. 把一批 **原始 PDF** 转成带页码标记（`<!-- page: N -->`）的 Markdown，中间产物落盘、可复用；
2. 用 `superindex.md_ingest.index_markdown` 建库；
3. 读取 **JSON / JSONL 问题集**，规范化成 `superindex.batch.Question`；
4. 用本项目"原服务 agent 链路"（与 `superindex batch` 同一套 client / prefetch / `run_question`）逐题回答；
5. 整理保存：`summary.md`、`answers.csv`、`answers.xlsx`。

**前置条件**

- 在仓库根目录的 `.env` 里配置问答模型，至少 `SUPERINDEX_CHAT_MODEL`（以及对应的 base url / key，参照 `.env.example`）。**模型必须支持 tool calling。**
  只做步骤 1-3（转换、建库、读题）不需要模型；`INDEX_SUMMARY=True` 时还需要 `SUPERINDEX_INDEX_MODEL`。
- 依赖：`uv sync --group notebook`（`ipykernel`、`pandas`、`openpyxl` 不在主依赖里）。
- 只改下面的 **配置 cell**，其余 cell 无需改动；所有 cell 可重复运行（已转换的 PDF、已建好的库、已成功的题都会跳过）。

**命令行执行**：`uv run --group notebook jupyter nbconvert --to notebook --execute notebooks/batch_qa.ipynb --output-dir <某目录>`
（配置项可用 `SI_NB_*` 环境变量覆盖，见配置 cell）。

**文字层转出的落盘格式**（`PDF_EXTRACTOR="text-layer"`，默认）

- 每个 PDF 存成 `WORK_DIR/md/<PDF 文件名 stem>.md`：UTF-8 Markdown，**每页前有 `<!-- page: N -->` 页码标记**；`ADD_HEADINGS=True` 时在对应页的页码标记后带 `#` 章节标题。
- 旁边的 `<stem>.meta.json` 记录：来源 PDF、抽取方式、页数、是否补了标题、空白页（数量与页码）、时间、`status`。
- 这些 Markdown 可以**直接人工查看、修改**；改动后重新运行，步骤 2 会因内容变化自动重新建库（不用 `FORCE_EXTRACT`，那会用 PDF 重新覆盖你的修改）。
- **没有文字层**的 PDF（所有页文字都为空，如扫描件）不写 `.md`、不建库，只写 `meta.json`（`status: "skipped_no_text_layer"`）并登记到 `skipped_pdfs.json`；指向它的题目不会去问模型，而是在结果里记一条 `status=skipped_no_text_layer` 的错误记录。需要处理这类 PDF 请改用 `azure-di`。
- 默认**绝不调用 Azure**：即使环境里有 Azure 密钥，也只有显式设 `PDF_EXTRACTOR = "azure-di"` 才会用。

**产出物**（默认都在 `results/notebook/<问题集文件名>/` 下，整个 `results/` 已被 git 忽略）

| 路径 | 内容 |
|---|---|
| `md/` | PDF 转出的 Markdown 及同名 `*.meta.json`，格式见下 |
| `skipped_pdfs.json` | 被跳过的 PDF（无文字层的扫描件等）及原因 |
| `store/` | 索引库 |
| `runs/<时间戳>/results.jsonl` | 逐题完整记录，与 `superindex batch` 格式兼容 |
| `runs/<时间戳>/summary.md`、`answers.csv`、`answers.xlsx` | 总览、答案表（csv 为 utf-8-sig，Excel 可直接打开） |

In [ ]:
# 定位仓库根目录并切换过去（.env、results/ 都相对 cwd）。必须在 import superindex 之前做。
import json
import os
import re
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display

try:
    from tqdm import tqdm
except ImportError:  # 没有 tqdm 就退化为 print
    tqdm = None


def find_repo_root(start: Path) -> Path:
    """向上找含 pyproject.toml 的目录。"""
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file():
            return folder
    raise RuntimeError(f"从 {start} 向上找不到 pyproject.toml，请在仓库内运行本 Notebook")


ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)

from superindex.runtime import (
    ConfigError,
    LLMSettings,
    configure_litellm,
    load_env,
    resolve_instructions,
    set_offline_defaults,
)

set_offline_defaults()  # 必须早于任何 litellm 的 import
ENV_FILES = load_env()  # 读取 ROOT/.env；已设置的环境变量优先

from superindex import page_images, prefetch
from superindex.batch import (
    RESULTS_FILE,
    SUMMARY_FILE,
    Question,
    _from_item,
    _scope,
    judge_hits,
    read_results,
    run_question,
    score,
    write_summary,
)
from superindex.cli import make_client
from superindex.engine.local_store import DocStore
from superindex.extractors.backend import Extractor
from superindex.md_ingest import PAGE_MARKER_RE, index_markdown

print(f"仓库根目录: {ROOT}")
print(f".env      : {', '.join(map(str, ENV_FILES)) or '（未找到，仅使用进程环境变量）'}")

In [ ]:
# ───────────── 配置（所有可调项都在这一格）─────────────
# 每项都可用环境变量 SI_NB_<名字> 覆盖（方便命令行 nbconvert 执行），例如 SI_NB_LIMIT=3。
def _env(name: str, default: Any, cast=str) -> Any:
    raw = os.environ.get(f"SI_NB_{name}")
    return default if raw is None or raw.strip() == "" else cast(raw.strip())


def _bool(text: str) -> bool:
    return text.lower() in ("1", "true", "yes", "on")


# --- 输入 ---
DATASET_PATH = Path(_env("DATASET", "data/questions.jsonl")).expanduser()  # 问题文件：.json 或 .jsonl
PDF_DIR = Path(_env("PDF_DIR", "data/pdfs")).expanduser()                  # 原始 PDF 目录（递归查找 *.pdf）

# 数据集字段名映射：{规范名: 数据集里的字段名}。
# expected / doc / id 可缺省：值设为 None，或数据里没有该字段都行。
# doc 缺省 = 在全库范围提问。其余字段原样带进结果（Question.extra）。
FIELD_MAP = _env("FIELD_MAP", {"id": "id", "question": "question", "expected": "expected", "doc": "doc"},
                 json.loads)
# JSON 顶层是 {"questions": [...]} 或直接列表时无需设置；顶层是其它键名（如 {"items": [...]}）时填这个键名。
RECORDS_KEY: str | None = _env("RECORDS_KEY", None)

# --- PDF → Markdown ---
PDF_EXTRACTOR = _env("PDF_EXTRACTOR", "text-layer")  # "text-layer"：PDF 文字层（原始 PageIndex 做法，完全离线）；
                                                     # "azure-di"：Azure Document Intelligence（需 AZURE_DI_ENDPOINT / AZURE_DI_KEY）
ADD_HEADINGS = _env("ADD_HEADINGS", True, _bool)     # 仅 text-layer：用离线版面分析（flash）给 Markdown 补 # 章节标题
FORCE_EXTRACT = _env("FORCE_EXTRACT", False, _bool)  # True：已有 .md 也重新转换（改了 PDF_EXTRACTOR / ADD_HEADINGS 后需要）

# --- 中间产物目录 ---
WORK_DIR = Path(_env("WORK_DIR", Path("results") / "notebook" / DATASET_PATH.stem)).expanduser()
MD_DIR = WORK_DIR / "md"        # 转出的 Markdown
STORE = WORK_DIR / "store"      # 索引库
RUNS_DIR = WORK_DIR / "runs"    # 每次运行一个时间戳目录

# --- 建库 ---
INDEX_SUMMARY = _env("INDEX_SUMMARY", False, _bool)  # True：建库时用 LLM（SUPERINDEX_INDEX_MODEL）写节点摘要；False：不调 LLM

# --- 问答 ---
LIMIT: int | None = _env("LIMIT", None, int)    # 只跑前 N 题；None = 全部
TIMEOUT = _env("TIMEOUT", 300.0, float)         # 每题超时（秒）
CONCURRENCY = _env("CONCURRENCY", 1, int)       # 并发题数
RESUME = _env("RESUME", True, _bool)            # True：续跑最近一次运行目录（跳过已成功的题、重跑出错的题）；False：新开时间戳目录
PREFETCH: bool | None = _env("PREFETCH", None, _bool)  # 关键词检索前置：None = 沿用项目默认（SUPERINDEX_PREFETCH，默认开）
PREFETCH_K: int | None = _env("PREFETCH_K", None, int)  # 前置页数：None = 沿用项目默认（5）

print(f"问题集   : {DATASET_PATH}")
print(f"PDF 目录 : {PDF_DIR}")
print(f"工作目录 : {WORK_DIR}")
print(f"抽取方式 : {PDF_EXTRACTOR}（补标题：{ADD_HEADINGS if PDF_EXTRACTOR == 'text-layer' else '不适用'}）")

## 步骤 1：PDF → Markdown

逐个 PDF 转换，`md/` 下已有同名 `.md` 就跳过（`FORCE_EXTRACT=True` 强制重转）。每个 `.md` 旁写一个 `.meta.json`。

- `text-layer`：`Extractor(env={}).document_text(pdf)` 读文字层并写入 `<!-- page: N -->` 页码标记（`env={}` 保证绝不调用 Azure）。
  `ADD_HEADINGS=True` 时再用 `page_index_flash`（离线版面分析，不调 LLM）得到章节树，在对应页的页码标记之后插入 `#` 标题；
  flash 失败或得不到结构时只给警告并退回无标题版本。
  **所有页文字都为空**的 PDF（扫描件）不写 `.md`、不建库：只写 `meta.json`（`status: "skipped_no_text_layer"`）并登记到 `WORK_DIR/skipped_pdfs.json`，下次运行不再重复抽取；
  部分页为空的 PDF 照常处理，空页数量和页码记在 `meta.json` 里并在汇总表给出警告。单个 PDF 出问题都不影响其它文档。
- 汇总表的 `status`：`converted` 已转换 / `skipped_existing` 已有 .md 跳过 / `skipped_no_text_layer` 无文字层 / `failed` 失败。
- `azure-di`：直接复用 `extractors.azure_di.extract_corpus`（Azure 输出本身带标题和表格）。

In [ ]:
def list_pdfs(pdf_dir: Path) -> list[Path]:
    if not pdf_dir.is_dir():
        raise FileNotFoundError(f"PDF 目录不存在：{pdf_dir.resolve()}（请修改配置里的 PDF_DIR）")
    pdfs = sorted(p for p in pdf_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf")
    if not pdfs:
        raise FileNotFoundError(f"{pdf_dir.resolve()} 下没有 PDF 文件")
    return pdfs


def add_headings(md: str, pdf: Path) -> tuple[str, int, str | None]:
    """用 flash 离线版面分析给 md 补 # 标题。返回 (新 md, 标题数, 警告)；失败时原样返回 md 并给出警告。"""
    try:
        from superindex.engine.flash import page_index_flash

        tree = page_index_flash(str(pdf), summary=False, optimize=False)
    except Exception as exc:  # noqa: BLE001 - 失败只退回无标题版本
        return md, 0, f"flash 失败（{type(exc).__name__}: {exc}），已退回无标题版本"

    heads: dict[int, list[str]] = {}

    def walk(nodes: list[dict] | None, level: int) -> None:
        for node in nodes or []:
            title = " ".join(str(node.get("title") or "").split())
            page = node.get("start_index")
            # flash 在找不到结构时会退化成一页一个 "Page N" 节点，那不是真标题
            if title and isinstance(page, int) and not re.fullmatch(r"Page \d+", title):
                heads.setdefault(page, []).append("#" * min(level, 6) + " " + title)
            walk(node.get("nodes"), level + 1)

    walk(tree.get("structure"), 1)
    if not heads:
        return md, 0, f"flash 未得到章节结构（toc_source={tree.get('toc_source')}），已退回无标题版本"

    parts = PAGE_MARKER_RE.split(md)  # [前导文本, 页码1, 页文本1, 页码2, 页文本2, ...]
    out, count = [parts[0]], 0
    for i in range(1, len(parts), 2):
        page, text = int(parts[i]), parts[i + 1]
        out.append(f"<!-- page: {page} -->")
        if page in heads:
            out.append("\n\n" + "\n\n".join(heads[page]) + "\n\n" + text.lstrip("\n"))
            count += len(heads[page])
        else:
            out.append(text)
    return "".join(out), count, None


SKIPPED = "skipped_no_text_layer"  # meta.json 里的 status：无文字层，已跳过


def meta_path_of(pdf: Path) -> Path:
    return (MD_DIR / pdf.relative_to(PDF_DIR)).with_suffix(".meta.json")


def write_meta(pdf: Path, **fields: Any) -> None:
    meta = {"source": str(pdf.resolve()), "extractor": PDF_EXTRACTOR, **fields,
            "extracted_at": datetime.now().isoformat(timespec="seconds")}
    path = meta_path_of(pdf)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")


def read_meta(pdf: Path) -> dict[str, Any]:
    try:
        return json.loads(meta_path_of(pdf).read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return {}


def extract_text_layer(pdf: Path, md_path: Path) -> dict[str, Any]:
    """一个 PDF 走文字层转 Markdown，写 .md 与 .meta.json；返回汇总表的一行。"""
    md = Extractor(env={}).document_text(pdf)  # env={} -> 即使环境里有 Azure 密钥也绝不调用 Azure
    parts = PAGE_MARKER_RE.split(md)
    pages = [(int(parts[i]), parts[i + 1]) for i in range(1, len(parts), 2)]
    empty_pages = [n for n, text in pages if not text.strip()]
    if not pages or len(empty_pages) == len(pages):  # 所有页文字都为空 -> 无文字层，跳过并标记
        write_meta(pdf, status=SKIPPED, pages=len(pages), empty_pages=empty_pages,
                   reason="PDF 没有文字层（扫描件/纯图片），文字层抽取结果全为空；如需处理请改用 azure-di")
        return {"status": SKIPPED, "pages": len(pages), "headings": 0,
                "note": "警告：无文字层（扫描件？），已跳过、不建库；请改用 PDF_EXTRACTOR=\"azure-di\" 处理"}
    notes = [f"警告：{len(empty_pages)} 页无文字 {empty_pages}（可能是整页图片）"] if empty_pages else []
    headings = 0
    if ADD_HEADINGS:
        md, headings, warn = add_headings(md, pdf)
        if warn:
            notes.append("警告：" + warn)
    md_path.parent.mkdir(parents=True, exist_ok=True)
    md_path.write_text(md, encoding="utf-8")
    write_meta(pdf, status="ok", pages=len(pages), empty_pages=empty_pages,
               add_headings=ADD_HEADINGS, headings_added=bool(headings), heading_count=headings)
    return {"status": "converted", "pages": len(pages), "headings": headings, "note": "；".join(notes)}


pdfs = list_pdfs(PDF_DIR)
rows: list[dict[str, Any]] = []

if PDF_EXTRACTOR == "text-layer":
    for pdf in pdfs:
        md_path = (MD_DIR / pdf.relative_to(PDF_DIR)).with_suffix(".md")
        meta = read_meta(pdf)
        if not FORCE_EXTRACT and (md_path.exists() or meta.get("status") == SKIPPED):
            # 已有 .md，或已被标记为无文字层：不重复抽取（FORCE_EXTRACT=True 可强制重试）
            status = SKIPPED if meta.get("status") == SKIPPED and not md_path.exists() else "skipped_existing"
            note = ""
            if status == SKIPPED:
                note = "此前已标记为无文字层，不重复抽取；改用 azure-di 或设 FORCE_EXTRACT=True 重试"
            elif meta and (meta.get("extractor") != PDF_EXTRACTOR
                           or meta.get("add_headings") != ADD_HEADINGS):
                note = "已有 .md 的抽取方式/补标题与当前配置不同；需要按新配置重转请设 FORCE_EXTRACT=True"
            if meta.get("empty_pages") and status != SKIPPED:
                note = (note + "；" if note else "") + f"警告：{len(meta['empty_pages'])} 页无文字 {meta['empty_pages']}"
            rows.append({"pdf": pdf.name, "status": status, "pages": meta.get("pages"),
                         "headings": meta.get("heading_count"), "note": note})
            continue
        if FORCE_EXTRACT and md_path.exists():
            md_path.unlink()  # 重转：避免上一次的 .md 在新结果为"无文字层"时残留
        try:
            result = extract_text_layer(pdf, md_path)
        except Exception as exc:  # noqa: BLE001 - 单个 PDF 失败不中断其它文档
            result = {"status": "failed", "pages": None, "headings": None,
                      "note": f"{type(exc).__name__}: {exc}"}
        rows.append({"pdf": pdf.name, **result})
elif PDF_EXTRACTOR == "azure-di":
    # 只有显式选了 azure-di 才会调用 Azure（需 AZURE_DI_ENDPOINT / AZURE_DI_KEY）
    from superindex.extractors.azure_di import AzureDIConfig, AzureDIError, extract_corpus

    try:
        AzureDIConfig.from_env().validate()  # 缺 AZURE_DI_ENDPOINT / AZURE_DI_KEY 时给出明确提示
    except AzureDIError as exc:
        raise RuntimeError(f"PDF_EXTRACTOR=\"azure-di\" 但 Azure 未配置：{exc}\n"
                           "或者把 PDF_EXTRACTOR 改回 \"text-layer\"（离线）。") from exc
    statuses: dict[str, str] = {}
    summary = extract_corpus(PDF_DIR, MD_DIR, force=FORCE_EXTRACT, workers=2, page_markers=True,
                             on_progress=lambda i, n, pdf, status, info: statuses.update({str(pdf): status}))
    errors = dict(e.split(": ", 1) for e in summary["errors"])
    for pdf in pdfs:
        status = {"converted": "converted", "skipped": "skipped_existing"}.get(statuses.get(str(pdf)), "failed")
        rows.append({"pdf": pdf.name, "status": status, "pages": read_meta(pdf).get("pages"), "headings": None,
                     "note": errors.get(pdf.name, "")})
else:
    raise ValueError(f"PDF_EXTRACTOR 只能是 \"text-layer\" 或 \"azure-di\"，当前是 {PDF_EXTRACTOR!r}")

# 汇总所有被跳过的 PDF（含此前运行标记的），写到 WORK_DIR/skipped_pdfs.json
skipped_pdfs = [{"pdf": str(p.relative_to(PDF_DIR)), "source": str(p.resolve()), "status": SKIPPED,
                 "reason": read_meta(p).get("reason"), "pages": read_meta(p).get("pages"),
                 "marked_at": read_meta(p).get("extracted_at")}
                for p in pdfs if read_meta(p).get("status") == SKIPPED]
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "skipped_pdfs.json").write_text(json.dumps(skipped_pdfs, ensure_ascii=False, indent=2), encoding="utf-8")

display(pd.DataFrame(rows))
for row in rows:
    if row["status"] in (SKIPPED, "failed") or "警告" in row["note"]:
        print(f"[{row['status']}] {row['pdf']}: {row['note']}")
print(f"被跳过的 PDF：{len(skipped_pdfs)} 个，清单见 {WORK_DIR / 'skipped_pdfs.json'}")

## 步骤 2：建库

对 `md/` 下每个 Markdown 调 `index_markdown`（`pdf=` 记录源 PDF 关联）。内容没变会自动跳过；
`INDEX_SUMMARY=False` 时不调 LLM。

In [ ]:
settings = LLMSettings.resolve()
summary_model = None
if INDEX_SUMMARY:
    summary_model = settings.require("index")  # 没配 SUPERINDEX_INDEX_MODEL 会给出明确提示
    configure_litellm()

md_files = sorted(MD_DIR.rglob("*.md")) if MD_DIR.is_dir() else []
if not md_files:
    raise RuntimeError(f"{MD_DIR} 下没有 Markdown，请先成功运行步骤 1")
pdf_by_stem = page_images.pdf_index(PDF_DIR)

index_rows: list[dict[str, Any]] = []
for md_path in md_files:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res = index_markdown(md_path, STORE, summary_model=summary_model,
                                 backend=settings.index_backend(), force=False,
                                 pdf=page_images.find_pdf(md_path, pdf_by_stem))
        index_rows.append({"doc_id": res.doc_id, "文档": res.name, "页数": res.pages, "节点数": res.nodes,
                           "状态": "跳过（未变）" if res.skipped else "已建库",
                           "页码标记": res.has_markers, "警告": "；".join(res.warnings)})
    except Exception as exc:  # noqa: BLE001 - 单个文档失败不中断其它文档
        index_rows.append({"doc_id": None, "文档": md_path.name, "页数": None, "节点数": None,
                           "状态": f"失败：{type(exc).__name__}: {exc}", "页码标记": None, "警告": ""})
display(pd.DataFrame(index_rows))

## 步骤 3：读问题

按 `FIELD_MAP` 把数据集规范化成 `batch.Question`（没有 id 时自动编号 `Q001…`；其余字段进 `extra`），
再检查每题的 `doc` 能否在库里匹配到（复用 `batch._scope`，含用 PDF 文件名 stem 匹配）。匹配不到的题仍会运行，在结果里记为错误。

In [ ]:
# results.jsonl 自己占用的字段名；数据集里与之同名的额外字段会被加前缀 src_，避免覆盖结果字段
STANDARD_KEYS = {"id", "question", "doc", "expected", "scope", "answer", "error", "seconds", "llm_turns",
                 "tool_calls", "pages_read", "score", "prefetch", "prefetch_hit", "page_images", "image_count", "status"}


def load_records(path: Path, records_key: str | None) -> list[Any]:
    """读 .json（顶层列表，或 {"questions": [...]} / {RECORDS_KEY: [...]}）或 .jsonl。"""
    if not path.is_file():
        raise FileNotFoundError(f"问题文件不存在：{path.resolve()}（请修改配置里的 DATASET_PATH）")
    text = path.read_text(encoding="utf-8-sig")
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix != ".json":
        raise ValueError(f"只支持 .json / .jsonl，当前是 {path.suffix!r}")
    data = json.loads(text)
    if isinstance(data, list):
        return data
    if not isinstance(data, dict):
        raise TypeError(f"{path.name}: 顶层应是列表或对象")
    key = records_key or ("questions" if "questions" in data else None)
    if key is None or not isinstance(data.get(key), list):
        candidates = [k for k, v in data.items() if isinstance(v, list)]
        raise ValueError(f"{path.name}: 顶层是对象，找不到题目列表；请设置 RECORDS_KEY，候选键：{candidates}")
    return data[key]


def to_questions(items: list[Any], field_map: dict[str, str | None]) -> list[Question]:
    """按字段映射规范化；字段名为 None 或数据里没有该字段都当作缺省。"""
    names = {k: field_map.get(k) for k in ("id", "question", "expected", "doc")}
    mapped = {v for v in names.values() if v}
    questions: list[Question] = []
    for index, item in enumerate(items, start=1):
        if isinstance(item, str) and names["question"]:
            item = {names["question"]: item}
        if not isinstance(item, dict):
            raise TypeError(f"第 {index} 条：应是对象，实际是 {item!r}")
        q = _from_item({k: item.get(v) for k, v in names.items() if v}, index)  # 复用 batch 的规范化与自动编号
        if q is None:
            print(f"  跳过第 {index} 条：问题为空")
            continue
        q.extra = {(f"src_{k}" if k in STANDARD_KEYS else k): v  # 其余字段原样带进结果
                   for k, v in item.items() if k not in mapped}
        questions.append(q)
    seen: set[str] = set()
    for q in questions:
        if q.id in seen:
            raise ValueError(f"问题 id 重复：{q.id!r}")
        seen.add(q.id)
    return questions


items = load_records(DATASET_PATH, RECORDS_KEY)
questions = to_questions(items, FIELD_MAP)
if not questions:
    first = items[0] if items else None
    raise ValueError(f"没有读到任何问题；请检查 FIELD_MAP[\"question\"]。数据第一条：{first!r}")
if LIMIT:
    questions = questions[:LIMIT]

store_docs = [m for m in DocStore(str(STORE)).list_metas() if m.get("status") == "completed"]
# 被标记为"无文字层"的 PDF（步骤 1 的 meta.json）
skipped_names = {Path(p["pdf"]).name: p for p in skipped_pdfs}


def skipped_pdf_of(q: Question) -> list[str]:
    """q.doc 的每一项都在库里匹配不到、且至少一项对应无文字层的 PDF 时，返回这些 PDF 的文件名。"""
    hit: list[str] = []
    for want in q.doc:
        try:
            _scope(store_docs, [want])
            return []  # 有项能在库里匹配到：照常提问
        except ConfigError:
            hit += [n for n in skipped_names if Path(want).stem.lower() in Path(n).stem.lower()]
    return hit


skipped_q: dict[str, str] = {}  # 题 id -> 跳过原因（这些题不会去问模型）
unmatched: list[tuple[str, list[str], str]] = []
for q in questions:
    if q.doc:
        if names := skipped_pdf_of(q):
            skipped_q[q.id] = f"跳过：源 PDF 无文字层（{', '.join(dict.fromkeys(names))}）"
            continue
        try:
            _scope(store_docs, q.doc)
        except ConfigError as exc:
            unmatched.append((q.id, q.doc, str(exc)))

print(f"题数 {len(questions)}（库内文档 {len(store_docs)} 个）；"
      f"{sum(1 for q in questions if q.doc)} 题指定了文档，{sum(1 for q in questions if not q.doc)} 题在全库范围提问")
if skipped_q:
    print(f"以下 {len(skipped_q)} 题指向无文字层的 PDF，不会去问模型，直接在结果里记为 skipped_no_text_layer：")
    for qid, why in skipped_q.items():
        print(f"  {qid}: {why}")
if unmatched:
    print(f"以下 {len(unmatched)} 题的 doc 在库里匹配不到（运行时会记为错误，请检查文档名或 PDF 是否已转换）：")
    for qid, doc, why in unmatched:
        print(f"  {qid}: {doc}")
display(pd.DataFrame([{"id": q.id, "question": q.question, "expected": q.expected,
                       "doc": "; ".join(q.doc), **q.extra} for q in questions]).head(10))

## 步骤 4：逐题回答

流程与 `superindex batch`（`cmd_batch`）对齐：同一个 `make_client`（含 `search_pages` / `calculate` 工具）、关键词检索前置（prefetch）、`run_question`，
记录字段与 `results.jsonl` 完全兼容，每完成一题立即追加写入；单题失败（含超时）只记在该题的 `error` 里，不中断整批。
`RESUME=True` 时续跑最近一次运行目录：跳过已成功的题，重跑出错的题。

In [ ]:
RUNS_DIR.mkdir(parents=True, exist_ok=True)
settings = LLMSettings.resolve()
instructions = resolve_instructions()

existing_runs = sorted(p for p in RUNS_DIR.glob("*") if (p / RESULTS_FILE).is_file())
run_dir = existing_runs[-1] if RESUME and existing_runs else RUNS_DIR / datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)
results_path = run_dir / RESULTS_FILE
results_path.touch(exist_ok=True)

done = {k: r for k, r in read_results(run_dir).items() if not r.get("error")} if RESUME else {}
todo = [q for q in questions if q.id not in done]
prefetch_k = prefetch.resolve_k(PREFETCH, PREFETCH_K)
image_mode = page_images.resolve_mode(None)
started = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"运行目录 : {run_dir}")
print(f"共 {len(questions)} 题：已完成 {len(questions) - len(todo)}，本次运行 {len(todo)}；"
      f"timeout {TIMEOUT:g}s，并发 {CONCURRENCY}，prefetch {prefetch_k or '关'}，page image {image_mode}")

client = None
if any(q.id not in skipped_q for q in todo):  # 只剩被跳过的题时不需要模型
    try:
        client = make_client(settings, STORE, instructions=instructions)  # 没配 SUPERINDEX_CHAT_MODEL 会在这里报错
    except ConfigError as exc:
        raise RuntimeError(f"未配置可用的问答模型：{exc}") from exc

lock = threading.Lock()
page_texts: dict[str, list[str]] = {}
bar = tqdm(total=len(todo), desc="回答问题", unit="题") if tqdm and todo else None


def say(text: str) -> None:
    (bar.write if bar else print)(text)


class _SkippedQuestion(Exception):
    """指向无文字层 PDF 的题：不问模型。"""


def answer_one(q: Question) -> None:
    try:
        if q.id in skipped_q:  # 指向无文字层 PDF：不问模型，只记一条说明
            raise _SkippedQuestion(skipped_q[q.id])
        scope_ids = _scope(store_docs, q.doc) if q.doc else None
        scope = scope_ids[0] if scope_ids and len(scope_ids) == 1 else scope_ids
        hits = prefetch.search(STORE, q.question, scope_ids, prefetch_k)
        session = page_images.new_session(STORE, image_mode)
        if session is not None:
            session.attach_prefetch(hits)
        record = run_question(client, q, scope, timeout=TIMEOUT, reasoning_effort=settings.reasoning_effort,
                              message=prefetch.augment(q.question, hits), session=session)
        if prefetch_k:
            judge, flags = judge_hits(STORE, q, hits, page_texts)
            record["prefetch"] = [{"doc_name": h.doc_name, "page": h.page, "relevant": f}
                                  for h, f in zip(hits, flags)]
            record["prefetch_hit"] = any(flags) if judge else None
    except Exception as exc:  # noqa: BLE001 - 如 doc 匹配不到：记下来，整批继续
        skipped = isinstance(exc, _SkippedQuestion)
        record = {"id": q.id, "question": q.question, "doc": q.doc, "expected": q.expected, **q.extra,
                  "scope": None, "answer": "", "error": str(exc) if skipped else f"{type(exc).__name__}: {exc}",
                  "seconds": 0.0, "llm_turns": 0, "tool_calls": [], "pages_read": [],
                  "score": score(q.expected, "")}
        if skipped:
            record["status"] = SKIPPED
    with lock:  # 每题完成立即追加写入
        with results_path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(record, ensure_ascii=False) + "\n")
        if bar:
            bar.update(1)
        status = (record["error"] if record.get("status") == SKIPPED else "ERROR " + record["error"]) if record["error"] else \
            ("命中" if (record["score"] or {}).get("hit") else ("未命中" if record["score"] else "已回答"))
        say(f"{q.id}  {record['seconds']:.1f}s  {status}  读取页: {', '.join(record['pages_read']) or '-'}")


t0 = time.time()
if todo:
    with ThreadPoolExecutor(max_workers=max(1, CONCURRENCY)) as pool:
        list(pool.map(answer_one, todo))
if bar:
    bar.close()
wall = time.time() - t0
print(f"本次运行 {len(todo)} 题，用时 {wall:.1f}s")

## 步骤 5：整理保存

`read_results` + `write_summary` 生成 `summary.md`；再用 pandas 导出 `answers.csv`（utf-8-sig，Excel 直接打开不乱码）和 `answers.xlsx`。

In [ ]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

latest = read_results(run_dir)
records = [latest[q.id] for q in questions if q.id in latest]
write_summary(records, run_dir / SUMMARY_FILE, {
    "started": started, "questions": str(DATASET_PATH), "store": str(STORE),
    "chat_model": settings.chat_model, "wall_seconds": wall, "prefetch_k": prefetch_k, "page_image": image_mode})

def flat(value: Any) -> Any:
    return json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value


def to_row(r: dict[str, Any]) -> dict[str, Any]:
    sc = r.get("score")
    row = {"id": r.get("id"), "question": r.get("question"), "answer": r.get("answer"),
           "expected": r.get("expected"), "score": ("跳过" if r.get("status") == SKIPPED else "出错" if r.get("error") else ("命中" if sc["hit"] else "未命中") if sc else None),
           "doc": "; ".join(r.get("doc") or []), "pages_read": ", ".join(r.get("pages_read") or []),
           "seconds": r.get("seconds"), "llm_turns": r.get("llm_turns"), "error": r.get("error"),
           "status": r.get("status") or ("error" if r.get("error") else "ok")}
    row.update({k: flat(v) for k, v in r.items() if k not in STANDARD_KEYS})  # 数据集带来的额外字段
    return row


df = pd.DataFrame([to_row(r) for r in records])
for col in ("expected", "score"):  # 数据集没有 expected 时不输出这两列
    if col in df and df[col].isna().all():
        df = df.drop(columns=col)

csv_path, xlsx_path = run_dir / "answers.csv", run_dir / "answers.xlsx"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.map(lambda v: ILLEGAL_CHARACTERS_RE.sub("", v)[:32767] if isinstance(v, str) else v) \
    .to_excel(xlsx_path, index=False)  # xlsx 不接受控制字符，单元格最长 32767 字符

skipped = sum(1 for r in records if r.get("status") == SKIPPED)
errors = sum(1 for r in records if r.get("error")) - skipped
scored = [r for r in records if r.get("score") and not r.get("error")]
hits = sum(1 for r in scored if r["score"]["hit"])
print(f"题数 {len(records)}：成功 {len(records) - errors - skipped}，失败 {errors}，跳过（无文字层）{skipped}"
      + (f"；粗评分命中 {hits}/{len(scored)}（{hits / len(scored):.0%}，仅供筛查，需人工复核）" if scored else ""))
print("产出文件：")
for path in (run_dir / RESULTS_FILE, run_dir / SUMMARY_FILE, csv_path, xlsx_path):
    print(f"  {path}")
display(df)